# 06 — Topic Modeling: BERTopic Pipeline vs LLM

Compares two approaches to topic modeling on the sustainable building corpus:

| Approach | Input | Method |
|----------|-------|--------|
| BERTopic pipeline | German text (original) | Unsupervised clustering + c-TF-IDF |
| LLM topic labeling | English translations | Zero-shot classification via Qwen2.5-7B |

**Design**:
- BERTopic runs on the **full corpus** (1,115 articles) — unsupervised, no labels needed
- LLM classifies articles into BERTopic-discovered topics using English translations
- Agreement between approaches measured on the 183-article overlap set
- Temporal evolution of topics tracked across 2015-2025

**Evaluation dimensions**:
1. Topic coherence (C_v score)
2. Topic diversity
3. Pipeline vs LLM agreement on the overlap set
4. Temporal stability of topics
5. Trade-off summary


In [ ]:
# Install dependencies
!pip install -q bertopic sentence-transformers umap-learn hdbscan gensim
!pip install -q huggingface_hub pandas numpy matplotlib seaborn scipy scikit-learn
!pip install -q 'numpy>=2.0'
print('Done')

In [ ]:
import json, pickle, time, re, warnings
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.metrics import cohen_kappa_score, confusion_matrix, classification_report
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')
plt.rcParams.update({'font.size': 11, 'figure.dpi': 130})
print('Imports OK')

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import yaml

PROJECT_ROOT = Path('/content/drive/MyDrive/thesis')
with open(PROJECT_ROOT / 'config.yaml') as f:
    config = yaml.safe_load(f)

SEED        = config.get('seed', 42)
DATA_PROC   = PROJECT_ROOT / 'Project' / 'Data' / 'Processed'
FIGURES_DIR = PROJECT_ROOT / 'Project' / 'Outputs' / 'Figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f'Config loaded | seed={SEED}')

In [ ]:
# Load corpus
df = pd.read_pickle(DATA_PROC / 'ner_pipeline_results.pkl')
df = df[~df['exclude']].copy()

# LLM checkpoint from notebook 04
with open(DATA_PROC / 'ner_llm_checkpoint.pkl', 'rb') as f:
    llm_ner_results = pickle.load(f)

ok_ids  = {aid for aid, r in llm_ner_results.items() if r['status'] == 'ok'}
df_val  = df[df['article_id'].isin(ok_ids)].copy()

docs_de = df['content'].fillna('').tolist()

print(f'Full corpus  : {len(df)} articles')
print(f'Overlap set  : {len(df_val)} articles (183 with LLM results)')

In [ ]:
# Embeddings (cached)
EMBED_CACHE = DATA_PROC / 'topic_embeddings.npy'
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

if EMBED_CACHE.exists():
    embeddings = np.load(EMBED_CACHE)
    print(f'Embeddings loaded from cache: {embeddings.shape}')
else:
    print('Computing embeddings (~5 min)...')
    embeddings = embedding_model.encode(docs_de, show_progress_bar=True, batch_size=32)
    np.save(EMBED_CACHE, embeddings)
    print(f'Embeddings computed and cached: {embeddings.shape}')

In [ ]:
# Fit BERTopic (cached)
BERTOPIC_CACHE = DATA_PROC / 'bertopic_model'

umap_model = UMAP(
    n_neighbors=15, n_components=5,
    min_dist=0.0, metric='cosine',
    random_state=SEED
)
hdbscan_model = HDBSCAN(
    min_cluster_size=15, min_samples=5,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

if BERTOPIC_CACHE.exists():
    topic_model = BERTopic.load(str(BERTOPIC_CACHE), embedding_model=embedding_model)
    topics      = topic_model.topics_
    probs       = topic_model.probabilities_
    print('BERTopic model loaded from cache')
else:
    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        language='multilingual',
        calculate_probabilities=True,
        verbose=True,
        nr_topics='auto',
    )
    topics, probs = topic_model.fit_transform(docs_de, embeddings)
    topic_model.save(
        str(BERTOPIC_CACHE), serialization='pickle',
        save_ctfidf=True, save_embedding_model=False
    )
    print('BERTopic fitted and saved')

topic_info = topic_model.get_topic_info()
n_topics   = len(topic_info[topic_info['Topic'] != -1])
n_outliers = sum(t == -1 for t in topics)

print(f'Topics found  : {n_topics}')
print(f'Outliers (-1) : {n_outliers} ({n_outliers/len(topics)*100:.1f}%)')
print(topic_info[topic_info['Topic'] != -1][['Topic', 'Count', 'Name']].head(15).to_string())

In [ ]:
# Topic coherence C_v
tokenised  = [doc.lower().split() for doc in docs_de]
dictionary = Dictionary(tokenised)

topic_words = []
for tid in topic_info[topic_info['Topic'] != -1]['Topic']:
    words = [w for w, _ in topic_model.get_topic(tid)[:10]]
    if words:
        topic_words.append(words)

coherence_model = CoherenceModel(
    topics=topic_words, texts=tokenised,
    dictionary=dictionary, coherence='c_v'
)
cv_score  = coherence_model.get_coherence()
all_words = [w for words in topic_words for w in words]
diversity = len(set(all_words)) / len(all_words) if all_words else 0

print(f'C_v coherence  : {cv_score:.4f}  (>0.5 is good)')
print(f'Topic diversity: {diversity:.4f}')
print(f'N topics       : {len(topic_words)}')

In [ ]:
# Assign topics to full corpus
df['bertopic_topic'] = topics
df['bertopic_prob']  = [p.max() if hasattr(p, '__len__') else float(p) for p in probs]
topic_name_map       = dict(zip(topic_info['Topic'], topic_info['Name']))
df['bertopic_name']  = df['bertopic_topic'].map(topic_name_map)

print('Topic distribution (top 10):')
print(df[df['bertopic_topic'] != -1]['bertopic_name'].value_counts().head(10).to_string())

In [ ]:
# Figure 1: Topic overview
topic_counts = topic_info[topic_info['Topic'] != -1].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('BERTopic: Topic Discovery on German Corpus (1,115 articles)',
             fontsize=13, fontweight='bold')

ax = axes[0]
ax.bar(range(len(topic_counts)), topic_counts['Count'].values,
       color='#2196F3', alpha=0.8, edgecolor='white')
ax.set_xlabel('Topic (ranked by size)')
ax.set_ylabel('Article count')
ax.set_title(f'{n_topics} topics discovered, {n_outliers} outliers')
ax.grid(axis='y', alpha=0.3)

ax2 = axes[1]
ax2.axis('off')
top5 = topic_info[topic_info['Topic'] != -1].head(5)
table_data = []
for _, row in top5.iterrows():
    words = ', '.join([w for w, _ in topic_model.get_topic(row['Topic'])[:5]])
    table_data.append([f"T{row['Topic']}", row['Count'], words])
table = ax2.table(cellText=table_data, colLabels=['Topic', 'n', 'Top keywords'],
                  loc='center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.auto_set_column_width([0, 1, 2])
ax2.set_title('Top 5 Topics by Size', pad=20)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'topic_bertopic_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 saved')

In [ ]:
# Figure 2: Temporal topic evolution
TOP_N = min(8, n_topics)
top_topic_ids = topic_info[topic_info['Topic'] != -1]['Topic'].head(TOP_N).tolist()

df_topics = df[df['bertopic_topic'].isin(top_topic_ids)].copy()
temporal  = (
    df_topics.groupby(['year_bin', 'bertopic_topic'])
    .size().unstack(fill_value=0)
)
temporal_norm = temporal.div(temporal.sum(axis=1), axis=0)
temporal_norm.columns = [topic_name_map.get(t, f'T{t}') for t in temporal_norm.columns]

fig, ax = plt.subplots(figsize=(12, 6))
temporal_norm.plot(kind='bar', ax=ax, colormap='tab10', alpha=0.85, edgecolor='white')
ax.set_title('Topic Prevalence by Year Bin (Top Topics, Full Corpus)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Year bin')
ax.set_ylabel('Proportion of articles')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(loc='upper right', fontsize=8, bbox_to_anchor=(1.3, 1))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'topic_temporal_evolution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 2 saved')
print(temporal_norm.round(3).to_string())

In [ ]:
# LLM topic classification setup
from google.colab import userdata
from huggingface_hub import InferenceClient

HF_TOKEN     = userdata.get('HF_TOKEN')
ACTIVE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'

topic_labels = {}
for tid in top_topic_ids:
    words = [w for w, _ in topic_model.get_topic(tid)[:3]]
    topic_labels[tid] = f"T{tid}: {', '.join(words)}"

TOPIC_LIST_STR = '\n'.join([f'  {v}' for v in topic_labels.values()])

TOPIC_SYSTEM_PROMPT = f"""You are a precise text classification system.
Classify the provided article into exactly ONE of these topic categories:

{TOPIC_LIST_STR}
  OTHER: does not fit any topic above

Rules:
- Choose the single best matching topic
- Respond with ONLY the topic ID (e.g. T0, T1, T2 ... or OTHER)
- No explanation, no punctuation, just the ID"""

client = None
for provider in ['together', 'nebius', 'fireworks-ai', 'sambanova']:
    try:
        _c    = InferenceClient(provider=provider, api_key=HF_TOKEN)
        _test = _c.chat_completion(
            model=ACTIVE_MODEL,
            messages=[{'role': 'user', 'content': 'Reply with: OK'}],
            max_tokens=5,
        )
        client = _c
        print(f'Provider: {provider} | Model: {ACTIVE_MODEL}')
        break
    except Exception as e:
        print(f'  {provider}: {e}')

if client is None:
    print('WARNING: No provider available. LLM classification cells will be skipped.')

print(f'\nTopic label set ({len(topic_labels)} topics):')
print(TOPIC_LIST_STR)

In [ ]:
# LLM inference loop
TOPIC_CHECKPOINT = DATA_PROC / 'topic_llm_checkpoint.pkl'
valid_ids        = {f'T{tid}' for tid in top_topic_ids}

def load_ckpt():
    if TOPIC_CHECKPOINT.exists():
        with open(TOPIC_CHECKPOINT, 'rb') as f:
            d = pickle.load(f)
        print(f'Checkpoint loaded: {len(d)} articles')
        return d
    return {}

def save_ckpt(d):
    with open(TOPIC_CHECKPOINT, 'wb') as f:
        pickle.dump(d, f)

def parse_topic(response, valid):
    r = response.strip().upper()
    if r in valid or r == 'OTHER':
        return r
    m = re.search(r'T(\d+)', r)
    if m:
        c = f'T{m.group(1)}'
        if c in valid:
            return c
    return 'OTHER'

topic_results = load_ckpt()
todo          = df_val[~df_val['article_id'].isin(topic_results.keys())]
print(f'Remaining: {len(todo)} articles')

if client is not None:
    status_counts = {'ok': 0, 'api_error': 0}
    for _, row in tqdm(todo.iterrows(), total=len(todo), desc='LLM Topics'):
        aid        = row['article_id']
        content_en = str(row.get('content_en', ''))[:4000]
        try:
            resp  = client.chat_completion(
                model=ACTIVE_MODEL,
                messages=[
                    {'role': 'system', 'content': TOPIC_SYSTEM_PROMPT},
                    {'role': 'user',   'content': f'Classify this article:\n\n{content_en}'},
                ],
                max_tokens=10,
                temperature=0.0,
            )
            raw   = resp.choices[0].message.content
            label = parse_topic(raw, valid_ids)
            topic_results[aid] = {'llm_topic': label, 'status': 'ok', 'raw': raw}
            status_counts['ok'] += 1
        except Exception as e:
            topic_results[aid] = {'llm_topic': 'ERROR', 'status': 'api_error', 'raw': str(e)}
            status_counts['api_error'] += 1

        if len(topic_results) % 10 == 0:
            save_ckpt(topic_results)
        time.sleep(1.0)

    save_ckpt(topic_results)
    print(f'Done: {status_counts}')
else:
    print('Skipped — no API provider available')

In [ ]:
# Merge and build comparison set
df_val = df_val.copy()
df_val['llm_topic']        = df_val['article_id'].map(
    lambda a: topic_results.get(a, {}).get('llm_topic'))
df_val['llm_topic_status'] = df_val['article_id'].map(
    lambda a: topic_results.get(a, {}).get('status'))

# Also pull bertopic assignments from df into df_val
df_val = df_val.merge(
    df[['article_id', 'bertopic_topic', 'bertopic_name', 'bertopic_prob']],
    on='article_id', how='left'
)
df_val['bertopic_topic_str'] = df_val['bertopic_topic'].apply(
    lambda t: f'T{t}' if t in top_topic_ids else 'OTHER'
)

df_compare = df_val[
    (df_val['llm_topic_status'] == 'ok') &
    (df_val['bertopic_topic'].isin(top_topic_ids))
].copy()

print(f'Comparison set: {len(df_compare)} articles')
print('BERTopic distribution:')
print(df_compare['bertopic_topic_str'].value_counts().to_string())
print('LLM distribution:')
print(df_compare['llm_topic'].value_counts().to_string())

In [ ]:
# Agreement metrics
exact_match = (df_compare['bertopic_topic_str'] == df_compare['llm_topic']).mean()
print(f'Exact match rate : {exact_match:.3f} ({exact_match*100:.1f}%)')
print(f'N articles       : {len(df_compare)}')

try:
    kappa = cohen_kappa_score(
        df_compare['bertopic_topic_str'],
        df_compare['llm_topic']
    )
    print(f"Cohen's Kappa    : {kappa:.3f}")
except Exception as e:
    kappa = None
    print(f"Cohen's Kappa    : could not compute ({e})")

all_labels = sorted(set(
    df_compare['bertopic_topic_str'].tolist() +
    df_compare['llm_topic'].tolist()
))
print('\nClassification report:')
print(classification_report(
    df_compare['bertopic_topic_str'],
    df_compare['llm_topic'],
    labels=all_labels, zero_division=0
))

In [ ]:
# Figure 3: Confusion matrix
cm = confusion_matrix(
    df_compare['bertopic_topic_str'],
    df_compare['llm_topic'],
    labels=all_labels
)
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_norm, annot=True, fmt='.2f',
            xticklabels=all_labels, yticklabels=all_labels,
            cmap='Blues', vmin=0, vmax=1,
            linewidths=0.5, ax=ax)
ax.set_xlabel('LLM predicted topic')
ax.set_ylabel('BERTopic reference topic')
ax.set_title('Topic Agreement: BERTopic vs LLM (row-normalised)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'topic_agreement_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 3 saved')

In [ ]:
# Translation quality effect
df_compare = df_compare.copy()
df_compare['correct'] = (
    df_compare['bertopic_topic_str'] == df_compare['llm_topic']
).astype(int)

high_acc = df_compare[df_compare['translation_quality'] == 'high']['correct'].mean()
low_acc  = df_compare[df_compare['translation_quality'] == 'low']['correct'].mean()
print(f'Agreement by translation quality:')
print(f'  High: {high_acc:.3f}')
print(f'  Low : {low_acc:.3f}')

h = df_compare[df_compare['translation_quality'] == 'high']['correct']
l = df_compare[df_compare['translation_quality'] == 'low']['correct']
tq_pval = None
if len(h) > 0 and len(l) > 0:
    _, tq_pval = stats.mannwhitneyu(h, l, alternative='two-sided')
    print(f'  Mann-Whitney p={tq_pval:.4f} {"*" if tq_pval < 0.05 else "ns"}')

In [ ]:
# Trade-off summary
tradeoff = pd.DataFrame([
    {
        'Approach'            : 'BERTopic (DE)',
        'Input language'      : 'German (original)',
        'Translation required': 'No',
        'Topic discovery'     : 'Unsupervised',
        'C_v coherence'       : round(cv_score, 3),
        'Reproducibility'     : 'High (fixed seed)',
        'Transparency'        : 'High (keywords visible)',
        'Cost'                : 'Free',
        'German proficiency'  : 'Not required',
    },
    {
        'Approach'            : 'LLM classification (EN)',
        'Input language'      : 'English (translated)',
        'Translation required': 'Yes (OPUS-MT)',
        'Topic discovery'     : 'Guided (BERTopic labels)',
        'C_v coherence'       : 'N/A',
        'Reproducibility'     : 'Medium (API variability)',
        'Transparency'        : 'Low (black-box)',
        'Cost'                : 'Free tier (credit limits)',
        'German proficiency'  : 'Not required',
    },
])
print(tradeoff.set_index('Approach').to_string())

In [ ]:
# Save all results
df[['article_id', 'source', 'year', 'year_bin',
    'bertopic_topic', 'bertopic_name', 'bertopic_prob']].to_csv(
    DATA_PROC / 'topic_bertopic_assignments.csv', index=False
)

df_compare[['article_id', 'source', 'year_bin', 'translation_quality',
            'bertopic_topic_str', 'llm_topic', 'correct']].to_csv(
    DATA_PROC / 'topic_comparison_results.csv', index=False
)

summary = {
    'n_articles_full_corpus'  : int(len(df)),
    'n_topics_discovered'     : int(n_topics),
    'n_outliers'              : int(n_outliers),
    'outlier_rate'            : round(n_outliers / len(df), 3),
    'cv_coherence'            : round(float(cv_score), 4),
    'topic_diversity'         : round(float(diversity), 4),
    'n_comparison_articles'   : int(len(df_compare)),
    'llm_exact_match_rate'    : round(float(exact_match), 4),
    'cohens_kappa'            : round(float(kappa), 4) if kappa is not None else None,
    'tq_effect_pval'          : round(float(tq_pval), 4) if tq_pval is not None else None,
}

with open(DATA_PROC / 'topic_modeling_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('All results saved')
print(json.dumps(summary, indent=2))

## Notebook summary

| Item | Detail |
|------|--------|
| BERTopic input | Full corpus, German, multilingual embeddings |
| LLM input | 183-article overlap set, English translations |
| Coherence | C_v via gensim (top-10 words per topic) |
| Agreement | Exact match + Cohen's Kappa |
| Saved | `topic_bertopic_assignments.csv`, `topic_comparison_results.csv`, `topic_modeling_summary.json` |
| Figures | Topic overview, temporal evolution, confusion matrix |

**Next**: `07_final_comparison.ipynb` — cross-task summary, final trade-off table, thesis conclusions.
